<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller: La Papelería de Don Mauricio — Libro Ordenado y Estantes Rotulados 📒🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Práctico Evaluativo · Módulo 02
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/Para%20Dummies/02_Data_Warehouse_Data_Lake_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🎈 ¿Qué vamos a hacer en este taller?

Don Mauricio tiene una **papelería escolar**. Apunta las ventas en un cuaderno con algunos errores y, además, guarda los reportes de cada mes en una **bodega** con estantes. Es la diferencia entre un **almacén de datos** (libro ordenado) y un **lago** (estantes con archivos). Vas a:

1. 📒 **Pasar el cuaderno al libro ordenado:** separar las ventas con errores (cuarentena) con un **recibo** que cierra, armar las **fichas** y los **renglones con códigos** (la estrella) y **resumir sin que el total cambie** (el cubo).
2. 🗄️ **Buscar en los estantes rotulados:** si cada estante dice el mes (`mes=03`), para saber lo de marzo solo abres los archivos de marzo, no los de toda la bodega (poda de particiones).
3. ✍️ **Contar con tus palabras** cuándo conviene la tienda ordenada y cuándo la plaza de mercado.

> 🧸 **Cómo funciona:** cada reto trae un cuadro de código con huecos `...` y pistas 💡. Reemplaza los `...` por lo que corresponda. Debajo hay una **autoverificación** que te orienta con cariño. No traen soluciones: ¡inténtalo tú!

### 📋 Instrucciones

- Completa solo los huecos `...`; no cambies las celdas de preparación ni de autoverificación.
- Al final usa *Restart & Run All*: todo debe correr sin errores y sin `...` pendientes.
- Todo funciona sin internet (datos inventados). La bodega se crea en una carpeta temporal que se borra sola. Repasa si quieres: [El libro de cuentas de doña Rosa](../../02%20-%20Data%20Warehousing%20y%20Data%20Lake/Para%20Dummies/00_Data_Warehousing_Dummies.ipynb), [El centro de acopio de papa](../../02%20-%20Data%20Warehousing%20y%20Data%20Lake/Para%20Dummies/01_Data_Lake_Dummies.ipynb) y [Tienda ordenada o plaza de mercado](../../02%20-%20Data%20Warehousing%20y%20Data%20Lake/Para%20Dummies/02_DW_vs_Lake_Dummies.ipynb).

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos inventados
# ============================================================
import atexit
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# El cuaderno de ventas de don Mauricio (¡con dos errores!)
cuaderno = pd.DataFrame({
    "fecha":    ["2025-03-03", "2025-03-03", "2025-03-04", "2025-03-04", "2025-03-05", "2025-03-05",
                 "2025-03-06", "2025-03-06", "2025-03-07", "2025-03-07", "2025-03-08", "2025-03-08"],
    "producto": ["Cuaderno", "Lápiz", "Borrador", "Cuaderno", "Lápiz", "Resma",
                 "Cuaderno", "Borrador", "Lápiz", "Resma", "Cuaderno", "Borrador"],
    "cantidad": [3, 10, 2, 0, 5, 1, 2, -1, 8, 2, 1, 4],
    "precio":   [4000, 800, 600, 4000, 800, 15000, 4000, 600, 800, 15000, 4000, 600],
})
# Las fichas: una por producto, con su código y su categoría
fichas = pd.DataFrame({"codigo": [1, 2, 3, 4],
                       "producto": ["Cuaderno", "Lápiz", "Borrador", "Resma"],
                       "categoria": ["Papel", "Escritura", "Escritura", "Papel"]})

# La bodega: un estante por mes (carpeta mes=01, mes=02, mes=03) con 2 archivos cada uno
BODEGA = tempfile.mkdtemp(prefix="bodega_taller_")
atexit.register(shutil.rmtree, BODEGA, ignore_errors=True)
rng = np.random.default_rng(5)
for mes in (1, 2, 3):
    (Path(BODEGA) / f"mes={mes:02d}").mkdir()
    for parte in (1, 2):
        pd.DataFrame({"mes": mes, "producto": ["Cuaderno", "Lápiz", "Borrador", "Resma"],
                      "total": rng.integers(1, 20, 4) * 500}).to_csv(Path(BODEGA) / f"mes={mes:02d}" / f"ventas{parte}.csv", index=False)

print("Líneas en el cuaderno:", len(cuaderno), "| archivos en la bodega:", len(list(Path(BODEGA).glob("mes=*/*.csv"))))
display(cuaderno.head(3))

---
## Reto 1: Del cuaderno al libro ordenado 📒

Las ventas con `cantidad` de 0 o negativa son errores: **no se botan**, se guardan aparte en **cuarentena**. Con las buenas se arma la **estrella**: las **fichas** (cada producto con su código) y los **renglones** de ventas, que solo guardan el **código** en vez del nombre. Al final se resume por categoría (el **cubo**) y el total **no debe cambiar**.

**Pasos**
1. `buenas` (`cantidad > 0`) y `cuarentena` (las demás); `recibo = (entraron, buenas, cuarentena)` y debe cerrar: entraron = buenas + cuarentena.
2. `renglones`: las ventas buenas con el **código** del producto (únelas con `fichas` por `producto`) y las columnas `fecha`, `codigo`, `cantidad`, `precio`, más `total = cantidad * precio`.
3. `total_general`: la suma de `total`. `por_categoria`: el total por categoría (une `renglones` con `fichas` por `codigo` y agrupa por `categoria`).

**Puntaje sugerido:** 40 puntos.

In [ ]:
# Casillas que debes llenar (empiezan vacías; no las borres)
buenas = cuarentena = recibo = renglones = total_general = por_categoria = None

# 💡 1) Separar lo bueno de lo malo (¡sin botar nada!) y llevar el recibo
buenas = cuaderno[cuaderno["cantidad"] > ...]
cuarentena = cuaderno[cuaderno["cantidad"] <= ...]
recibo = (len(cuaderno), len(...), len(cuarentena))

# 💡 2) Renglones con código: unir con las fichas por el nombre del producto y quedarse con las columnas pedidas
renglones = buenas.merge(fichas[["codigo", "producto"]], on="...")[["fecha", "codigo", "cantidad", "precio"]]
renglones["total"] = renglones["cantidad"] * renglones["..."]

# 💡 3) El total y el cubo por categoría
total_general = renglones["total"].sum()
por_categoria = renglones.merge(fichas[["codigo", "categoria"]], on="codigo").groupby("...")["total"].sum()

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert buenas is not None and cuarentena is not None and len(buenas) == 10 and len(cuarentena) == 2, "Pista 🧸: son buenas las ventas con cantidad > 0 (10) y van a cuarentena las de cantidad <= 0 (2)."
assert recibo == (12, 10, 2) and recibo[0] == recibo[1] + recibo[2], "Pista 🧸: recibo = (len(cuaderno), len(buenas), len(cuarentena)) y debe cerrar: 12 = 10 + 2."
assert renglones is not None and list(renglones.columns) == ["fecha", "codigo", "cantidad", "precio", "total"], "Pista 🧸: renglones debe tener las columnas fecha, codigo, cantidad, precio y total (el nombre del producto ya no va: queda en las fichas)."
assert len(renglones) == 10 and set(renglones["codigo"]) <= {1, 2, 3, 4}, "Pista 🧸: debe haber un renglón por venta buena (10) y cada uno trae el código del producto."
assert int(renglones["total"].sum()) == 91000 and total_general == 91000, "Pista 🧸: total = cantidad * precio en cada renglón; el total general de las ventas buenas es 91 000."
assert por_categoria is not None and dict(por_categoria) == {"Escritura": 22000, "Papel": 69000}, "Pista 🧸: agrupa por categoría: Papel 69 000 y Escritura 22 000."
assert por_categoria.sum() == total_general, "Al resumir por categoría el total NO cambia: 69 000 + 22 000 = 91 000."
print(f"✅ ¡Reto 1 superado! Recibo {recibo[0]} = {recibo[1]} + {recibo[2]} | total {total_general:,} | por categoría {({k: int(v) for k, v in por_categoria.items()})} (suma igual al total)")

---
## Reto 2: Los estantes rotulados por mes 🗄️

En la bodega, cada **estante** lleva un rótulo con el mes (`mes=01`, `mes=02`, `mes=03`) y tiene 2 archivos. Si te piden «lo de marzo», **no hace falta abrir todos los archivos**: basta con los del estante `mes=03`. Eso se llama **poda de particiones**. El resultado debe ser el mismo que abrir todo y filtrar, pero con menos trabajo.

**Pasos**
1. `todos`: lista de **todos** los archivos (`BODEGA/mes=*/*.csv`, con `Path(BODEGA).glob(...)`). `solo_marzo`: solo los del estante `mes=03`.
2. `total_podado`: suma de la columna `total` leyendo **solo** los archivos de marzo.
3. `total_completo`: suma de `total` leyendo **todos** los archivos y filtrando `mes == 3`.
4. `archivos_abiertos`: cuántos archivos abriste en la versión podada.

**Puntaje sugerido:** 35 puntos.

In [ ]:
# Casillas que debes llenar
todos = solo_marzo = total_podado = total_completo = archivos_abiertos = None

# 💡 1) Los archivos: el comodín * significa «cualquiera»
todos = sorted(Path(BODEGA).glob("mes=*/*.csv"))
solo_marzo = sorted(Path(BODEGA).glob("mes=.../*.csv"))     # el estante de marzo se llama mes=03

# 💡 2) Leer SOLO los archivos de marzo y sumar
df_marzo = pd.concat([pd.read_csv(archivo) for archivo in ...])
total_podado = df_marzo["total"].sum()

# 💡 3) Leer los 6 archivos y filtrar después
df_todo = pd.concat([pd.read_csv(archivo) for archivo in todos])
total_completo = df_todo[df_todo["mes"] == ...]["total"].sum()

# 💡 4) ¿Cuántos archivos abriste en la versión podada?
archivos_abiertos = len(...)

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert todos is not None and len(todos) == 6, "Pista 🧸: Path(BODEGA).glob('mes=*/*.csv') encuentra los 6 archivos (3 estantes × 2 archivos)."
assert solo_marzo is not None and len(solo_marzo) == 2 and all(p.parent.name == "mes=03" for p in solo_marzo), "Pista 🧸: el estante de marzo se llama mes=03; son 2 archivos."
_esp = int(pd.concat([pd.read_csv(p) for p in todos]).query("mes == 3")["total"].sum())
assert total_podado is not None and int(total_podado) == _esp, "Pista 🧸: suma la columna 'total' leyendo solo los archivos de solo_marzo."
assert total_completo is not None and int(total_completo) == _esp, "Pista 🧸: lee todos los archivos, filtra las filas con mes == 3 y suma 'total'."
assert int(total_podado) == int(total_completo), "La poda no cambia el resultado: leer solo marzo y leer todo y filtrar dan la misma cifra."
assert archivos_abiertos == 2 < len(todos), "Pista 🧸: con la poda abriste 2 archivos en vez de 6."
print(f"✅ ¡Reto 2 superado! Marzo suma {int(total_podado):,} abriendo {archivos_abiertos} de {len(todos)} archivos; leer todo da la misma cifra pero con {len(todos) // archivos_abiertos} veces más archivos.")

---
## Reto 3: ¿Tienda ordenada o plaza de mercado? ✍️

**Puntaje sugerido:** 25 puntos (sin código). Escribe **3 a 5 líneas** en la celda de abajo respondiendo:

- ¿Por qué mandaste a **cuarentena** las ventas con cantidad 0 o negativa en vez de borrarlas? ¿Qué te dice el recibo `12 = 10 + 2`?
- ¿Por qué las **fichas** y los **códigos** ayudan a resumir (por ejemplo, por categoría) sin que el total cambie?
- ¿Qué te ahorra la **poda** de los estantes y por qué solo sirve si el rótulo (`mes=03`) coincide con lo que preguntas? Si llega un reporte con una columna nueva, ¿lo recibiría mejor la tienda ordenada o la plaza de mercado?

---
#### ✍️ Tu respuesta

_Escribe aquí tus 3-5 líneas. Reemplaza este texto por tu reflexión._

---
### 🧠 Rúbrica corta

| Criterio | Puntos |
|---|---|
| Reto 1: cuarentena, recibo que cierra, fichas y renglones con código, cubo con total igual | 40 |
| Reto 2: estantes rotulados, poda de particiones y mismo resultado que leer todo | 35 |
| Reto 3: reflexión con tus cifras | 25 |
| **Total** | **100** |

**Checklist:** ✅ ejecuté *Run All* sin errores · ✅ no quedan `...` · ✅ las dos autoverificaciones imprimen ✅ · ✅ escribí mi reflexión.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>